In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.2


In [2]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


In [3]:
# 1. Profile the data — find the problems

In [4]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [5]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [6]:
# LAB EXERCISE 1 — Profile it yourself

In [7]:
# -----------------------------------------------------------
# 🔹 Data Quality Check
# -----------------------------------------------------------

# 1. Duplicate row count

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 1


In [8]:
# 2. Missing values per column

missing_count = df.isnull().sum()

print("\nMissing values per column:")
print(missing_count)


Missing values per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [9]:
# 3. Problems I can see:
# - Duplicate rows may exist and need to be removed.
# - Missing values may need handling (remove or impute).
# - Numerical columns may contain outliers.
# - Categorical columns may have inconsistent categories.
# - Some columns may have different scales and require preprocessing.

In [10]:
# 2. Missing values — detect & handle

In [12]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------
import pandas as pd
import numpy as np

# Example data (if df is not already created)
df = pd.DataFrame({
    'age': [25, 30, -1, 40, 22],
    'spend': ['1000', 'N/A', '500', 'N/A', '800']
})

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

Missing after unmasking:
age      1
spend    2
dtype: int64


In [13]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after imputing: 0


In [14]:
# LAB EXERCISE 2 — Compare drop vs impute

In [16]:
import pandas as pd
import numpy as np

# Fresh copy
ex = raw.copy()

# 1. Unmask missing values

# Convert spend to numeric ('N/A' -> NaN)
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')

# Replace sentinel value -1 with NaN in age
ex['age'] = ex['age'].replace(-1, np.nan)

In [18]:
# 2a. Drop missing values

dropna_version = ex.dropna()
# 2b. Median-impute version

impute_version = ex.copy()

impute_version['age'] = impute_version['age'].fillna(
    impute_version['age'].median()
)

impute_version['spend'] = impute_version['spend'].fillna(
    impute_version['spend'].median()
)

In [19]:
# 3. Compare row counts

original_rows = len(ex)
dropna_rows = len(dropna_version)
imputed_rows = len(impute_version)

print("Original rows:", original_rows)
print("Rows after dropna():", dropna_rows)
print("Rows after median imputation:", imputed_rows)

print("\nRows lost by dropping:",
      original_rows - dropna_rows)

Original rows: 8
Rows after dropna(): 5
Rows after median imputation: 8

Rows lost by dropping: 3


In [20]:
# 3. Duplicates & data types

In [21]:
# -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (5, 2)
After : (5, 2) -> removed the repeated Gus row


In [25]:
# LAB EXERCISE 3 — Dedupe & retype

In [26]:
import pandas as pd

# Fresh copy
ex = raw.copy()

# 1. Fix data types

# Convert spend to numeric ('N/A' becomes NaN)
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')

# Convert date to datetime
ex['date'] = pd.to_datetime(ex['date'], errors='coerce')


# 2. Drop duplicate rows

ex = ex.drop_duplicates()


# 3. Print dtypes and shape

print("Data Types:")
print(ex.dtypes)

print("\nFinal Shape:")
print(ex.shape)

Data Types:
id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object

Final Shape:
(7, 6)


In [27]:
# 4. Outliers — detect with the IQR rule

In [29]:
import pandas as pd
import numpy as np

# Example: use your cleaned DataFrame
# ex = raw.copy()

# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# Calculate Q1, Q3 and IQR
q1, q3 = ex['spend'].quantile([0.25, 0.75])

iqr = q3 - q1

# IQR bounds
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

print(f'Q1 = {q1:.1f}')
print(f'Q3 = {q3:.1f}')
print(f'IQR = {iqr:.1f}')

print(f'\nNormal range: {low:.1f} to {high:.1f}')

# Find outliers
outliers = ex[(ex['spend'] < low) | (ex['spend'] > high)]

print("\nOutlier rows:")
print(outliers[['name', 'spend']])

Q1 = 112.6
Q3 = 187.6
IQR = 75.0

Normal range: 0.1 to 300.2

Outlier rows:
  name    spend
5  Fin  99000.0


In [30]:
# LAB EXERCISE 4 — Find the outliers

In [31]:
# 1. Q1, Q3, IQR for 'age'

Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)

IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)


# 2. Lower & Upper Bounds

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("\nLower Bound:", lower_bound)
print("Upper Bound:", upper_bound)


# 3. Rows outside the bounds

age_outliers = df[
    (df['age'] < lower_bound) |
    (df['age'] > upper_bound)
]

print("\nRows with age outside IQR bounds:")
print(age_outliers)

Q1: 25.0
Q3: 30.0
IQR: 5.0

Lower Bound: 17.5
Upper Bound: 37.5

Rows with age outside IQR bounds:
    age  spend
3  40.0  800.0


In [32]:
# 5. Messy text & inconsistent categories

In [33]:
import pandas as pd

messy = pd.Series(
    [' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
    dtype='string'
)

# 1. strip + lower

clean = messy.str.strip().str.lower()


# 2. map 'n.y.' -> 'new york'

clean = clean.replace('n.y.', 'new york')


# 3. value_counts()

print(clean.value_counts())

london      3
new york    3
Name: count, dtype: Int64
